# Challenge 0 final submission: MovieLens-20M

This is the leader's integrated notebook. It uses the shared preprocessing and metric definitions, answers Challenge Questions 1–5, and keeps runtime feasibility separate from rating-year and release-year trends. Run it on the full MovieLens-20M data before submitting; the sample fallback is only a smoke test.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from load_data import load_movielens
from preprocess import build_shared_tables, prepare_genome_scores, prepare_genome_tags
from metrics import (
    activity_group_summary,
    data_quality_report,
    genre_pair_stats,
    genre_q1_table,
    genre_summary,
    ratings_by_year,
    rating_distribution,
    release_year_stats,
    runtime_feasibility,
    tag_genome_comparison,
    tag_summary,
    user_activity_summary,
)


In [ ]:
requested_data_dir = Path(os.environ.get('MOVIELENS_DATA_DIR', PROJECT_ROOT / 'data' / 'ml-20m'))
try:
    frames = load_movielens(requested_data_dir)
    USING_SAMPLE = False
except FileNotFoundError:
    frames = load_movielens(PROJECT_ROOT / 'data' / 'sample')
    USING_SAMPLE = True

ratings_raw = frames['ratings']
movies_raw = frames['movies']
tags_raw = frames.get('tags', pd.DataFrame(columns=['userId', 'movieId', 'tag', 'timestamp']))
tables = build_shared_tables(ratings_raw, movies_raw, tags_raw)
ratings = tables['ratings']
movies = tables['movies']
tags = tables['tags']
movie_stats = tables['movie_stats']
user_stats = tables['user_stats']
exploded_genres = tables['exploded_genres']
rating_genre_df = tables['rating_genre_df']
links = frames.get('links', pd.DataFrame())
genome_scores = frames.get('genome-scores', pd.DataFrame())
genome_tags = frames.get('genome-tags', pd.DataFrame())
genome_scores = prepare_genome_scores(genome_scores) if not genome_scores.empty else pd.DataFrame()
genome_tags = prepare_genome_tags(genome_tags) if not genome_tags.empty else pd.DataFrame()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'summary_tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Using sample fixture:', USING_SAMPLE)
print('Loaded data directory:', frames['_data_dir'])

## Dataset overview and quality

All downstream sections use the same prepared tables. Genre-attributed rating statistics count a multi-genre movie once per genre by design.

In [ ]:
quality = data_quality_report(
    frames,
    ratings,
    movies,
    tags,
    genome_scores if not genome_scores.empty else None,
    genome_tags if not genome_tags.empty else None,
)
quality['file_summary'].to_csv(OUTPUT_DIR / 'final_file_summary.csv', index=False)
quality['key_integrity'].to_csv(OUTPUT_DIR / 'final_key_integrity.csv', index=False)
display(quality['file_summary'])
display(quality['key_integrity'])
rating_counts, rating_summary = rating_distribution(ratings)
display(rating_summary)


## Challenge Question 1

**Which genre has the highest average rating, and does that change when only movies with at least 1000 ratings are considered?**

**Method:** genre means are calculated over rating records. The support filter is applied at the movie level before those records are aggregated.

In [ ]:
genre_stats = genre_summary(exploded_genres, rating_genre_df)
q1 = genre_q1_table(
    movie_stats,
    rating_genre_df,
    exploded_genres,
    min_movie_ratings=1000,
)
q1.to_csv(OUTPUT_DIR / 'challenge_q1_genre_comparison.csv', index=False)
display(q1)
print('Interpretation: report the top genre in each scope together with movie and rating support; do not equate a high mean with popularity.')

## Challenge Question 2

**Which users are most active, and how do their average ratings compare with less-active users?**

The comparison uses empirical activity groups so the conclusion is descriptive and does not depend on an arbitrary fixed threshold.

In [ ]:
top_active_users = user_stats.nlargest(20, 'rating_count')
activity_groups = activity_group_summary(user_stats)
top_active_users.to_csv(OUTPUT_DIR / 'challenge_q2_top_active_users.csv', index=False)
activity_groups.to_csv(OUTPUT_DIR / 'challenge_q2_activity_groups.csv', index=False)
display(top_active_users)
display(user_activity_summary(user_stats))
display(activity_groups)
print('Interpretation: compare average ratings and rating variability, but do not make a causal claim about why active users rate differently.')

## Challenge Question 3

**Which tags are most frequently used, and do frequent user tags have low or high genome relevance?**

Primary tag cleaning is lowercase plus surrounding whitespace removal. The genome comparison additionally normalizes hyphens to spaces and reports coverage explicitly.

In [ ]:
tag_frequency = tag_summary(tags)
tag_frequency.to_csv(OUTPUT_DIR / 'challenge_q3_tag_frequency.csv', index=False)
display(tag_frequency.head(30))
if not genome_scores.empty and not genome_tags.empty:
    tag_genome = tag_genome_comparison(tags, genome_scores, genome_tags)
    tag_genome.to_csv(OUTPUT_DIR / 'challenge_q3_user_tag_genome_comparison.csv', index=False)
    display(tag_genome.head(30))
    print('Exact normalized user-tag to genome-tag match rate:', tag_genome['genome_tag_id'].notna().mean() if len(tag_genome) else np.nan)
else:
    tag_genome = pd.DataFrame()
    print('Genome files are unavailable; Q3 frequency is available but relevance comparison is pending.')

## Challenge Question 4

**Is there a trend in movie runtime?**

MovieLens does not provide runtime in its core files. A valid trend requires external metadata joined through IMDb or TMDb identifiers. Title length is not used as a proxy.

In [ ]:
runtime_result = runtime_feasibility(links, movies)
display(pd.Series(runtime_result, name='value').to_frame())

## Challenge Question 5

**For multi-genre movies, does a genre pair receive a higher average rating than the individual genres alone?**

The primary result requires at least 50 movies per pair. Pair averages are compared with both the better individual genre average and the mean of the two individual genre averages. The individual genre averages include pair movies, so this is a descriptive overlap comparison rather than an independent causal test.

In [ ]:
q5_pairs = genre_pair_stats(
    ratings,
    movies,
    genre_stats,
    min_movie_count=50,
)
q5_pairs.to_csv(OUTPUT_DIR / 'challenge_q5_genre_pair_stats.csv', index=False)
display(q5_pairs.head(20))
if q5_pairs.empty:
    print('No pair meets the minimum-support rule in this run; do not promote a low-support pair to a finding.')

## Supporting temporal views

Rating year and release year answer different questions. The first describes when users submitted ratings; the second describes the movie metadata year. The release-year table uses a documented minimum rating-record support of 1000.

In [ ]:
rating_year_stats = ratings_by_year(ratings)
release_year_view = release_year_stats(ratings, movies, min_rating_count=1000)
rating_year_stats.to_csv(OUTPUT_DIR / 'rating_year_stats.csv', index=False)
release_year_view.to_csv(OUTPUT_DIR / 'release_year_stats.csv', index=False)
display(rating_year_stats.head())
display(release_year_view.head())

## Limitations and final handoff

- The full-data run is required for course findings; the tracked sample is only a smoke test.
- Ratings are observational and do not identify causal effects.
- Multi-genre attribution duplicates a movie's rating record across its genres.
- Older release years and incomplete final rating years have uneven support.
- Exact user-tag/genome-tag matching underestimates semantic agreement when wording differs.
- Runtime needs an external, reproducible metadata join.

Before submission, review the contributor notebooks, execute this notebook top-to-bottom on MovieLens-20M, check exported figures, and replace the parameterized language in final_writeup.md with the observed full-data values.